# Zytama studio model — training on Colab

Fine-tunes **U²-Netp** (the app's product cut-out model) on your reviewed photos and exports a `.tflite` the app can use unchanged.

**Before you start** (see `README.md`):
1. *Runtime → Change runtime type → **T4 GPU***.
2. In Google Drive create a folder **`zytama_studio`** and upload into it:
   - `studio_model/` — this whole folder (`ml/studio_model` from the repo)
   - `convert_u2netp_to_tflite.py` — from the repo's `scripts/`
   - `u2netp.tflite` — the app's current model, from `assets/models/` (for comparison)
   - `u2netp.pth` — the starting weights (link in the U-2-Net README)
   - `dataset/` — the reviewed dataset (`images/`, `masks/`, `rejected.txt`)
   - *(optional)* `backgrounds/` — photos of shelves, tables, counters, screens **without** products

Then run the cells top to bottom. Training takes roughly 1–3 hours for 1,000 photos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/zytama_studio'
import os
for need in ['studio_model', 'convert_u2netp_to_tflite.py', 'u2netp.tflite', 'u2netp.pth', 'dataset/images', 'dataset/masks']:
    print(('OK      ' if os.path.exists(f'{D}/{need}') else 'MISSING ') + need)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -r {D}/studio_model/requirements.txt
!git clone -q --depth 1 https://github.com/xuebinqin/U-2-Net.git /content/U-2-Net

## Optional: build the dataset here
Skip if you already ran `prepare_dataset.py` on your computer and uploaded `dataset/`. Otherwise upload your raw photos to `zytama_studio/photos/` and run this; then **review** `dataset/review.html` (download it with the `review/` folder) before training.

In [ ]:
# !cd {D}/studio_model && python prepare_dataset.py --input {D}/photos --out {D}/dataset --teacher isnet

## Train

In [ ]:
RUN = f'{D}/runs/exp1'
BG = f'--backgrounds {D}/backgrounds' if os.path.isdir(f'{D}/backgrounds') else ''
!cd {D}/studio_model && python train.py --data {D}/dataset --repo /content/U-2-Net \
    --weights {D}/u2netp.pth --out {RUN} --epochs 60 --batch 16 --workers 2 {BG}

## Export to TFLite (same format the app uses)

In [ ]:
!python {D}/convert_u2netp_to_tflite.py --repo /content/U-2-Net \
    --weights {RUN}/best.pth --out {RUN}/u2netp_finetuned.tflite --work /content/tflite_work

## Compare with the app's current model
Read the **VERDICT** line. Only ship when it says SHIP.

In [ ]:
!cd {D}/studio_model && python evaluate.py --data {D}/dataset --old {D}/u2netp.tflite \
    --new {RUN}/u2netp_finetuned.tflite --out {RUN}/eval

## Ship it
If the verdict is **SHIP**: download `runs/exp1/u2netp_finetuned.tflite` from Drive, rename it to `u2netp.tflite`, replace `assets/models/u2netp.tflite` in the app, and do a full rebuild. No code changes are needed.

Keep the old file (e.g. `u2netp_v1.tflite` outside the app) so you can roll back.